In [ ]:
#!/usr/bin/env python3
"""
Aggressive |t| binning optimizer for Upsilon exclusive studies.
Converted to run natively inside a SWAN Jupyter Notebook cell.

Main features
-------------
1. Uses the same baseline selection as the kinematic dashboard:
   reco_mass in [8, 12], |Sigma(E-pz)-2Ee| < 0.2 Ee, reco_W >= 20.

2. Compares all |t| estimators:
   eX, BABE, eXBABE, proj.

3. Treats projected-t consistently:
   - stat_weight = weight
   - corrected_weight = weight_proj
   so the wedge correction does not artificially satisfy statistics cuts.

4. Scans raw and calibrated reco |t| definitions.
   Calibration modes:
   - none
   - scale: robust global scale factor from true/reco
   - quantile: monotonic MC-derived quantile mapping reco -> true

5. Optionally reads a previous t_resolution_summary.csv and attaches
   bias/core-resolution diagnostics to each candidate bin.

The script does NOT decide that a failing binning is a final recommendation.
If nothing passes, it reports a diagnostic fallback and advises asking theory
for a fine grid that can be rebinned later.
"""

import argparse
import os
import sys
import itertools
import numpy as np
import pandas as pd
import ROOT

ROOT.gROOT.SetBatch(True)

BASE_EDGES_DEFAULT = [0, 0.02, 0.05, 0.08, 0.12, 0.18, 0.25, 0.35, 0.50, 0.75, 1.0, 1.5, 2.0]

METHODS = {
    "eX": {
        "label": "lepton + Upsilon, t_eX",
        "reco": "reco_t_eX",
        "stat_weight": "weight",
        "corr_weight": "weight",
        "extra_cut": "",
        "preference": 1,
    },
    "BABE": {
        "label": "proton-only, t_BABE",
        "reco": "reco_t_BABE",
        "stat_weight": "weight",
        "corr_weight": "weight",
        "extra_cut": "",
        "preference": 3,
    },
    "eXBABE": {
        "label": "hybrid, t_eXBABE",
        "reco": "reco_t_eXBABE",
        "stat_weight": "weight",
        "corr_weight": "weight",
        "extra_cut": "",
        "preference": 4,
    },
    "proj": {
        "label": "projected-t wedge, proton-tagged hybrid baseline",
        "reco": "reco_t_proj",
        "stat_weight": "weight",
        "corr_weight": "weight_proj",
        "extra_cut": "weight_proj > 0 && reco_t_eXBABE > 0",
        "preference": 2,
    },
}

FIXED_CANDIDATES = {
    "diagnostic_fine": [0, 0.25, 0.50, 0.75, 1.0, 1.5, 2.0],
    "coarse_4bins": [0, 0.50, 1.0, 1.5, 2.0],
    "coarse_3bins": [0, 0.50, 1.0, 2.0],
    "very_coarse_2bins": [0, 0.75, 2.0],
    "ultra_coarse_2bins": [0, 1.0, 2.0],
}


def parse_edges(s):
    return [float(x) for x in s.split(",") if x.strip()]


def weighted_quantile(values, quantiles, sample_weight=None):
    values = np.asarray(values, dtype=float)
    quantiles = np.asarray(quantiles, dtype=float)
    if sample_weight is None:
        return np.quantile(values, quantiles)

    sample_weight = np.asarray(sample_weight, dtype=float)
    mask = np.isfinite(values) & np.isfinite(sample_weight) & (sample_weight > 0)
    values = values[mask]
    sample_weight = sample_weight[mask]
    if len(values) == 0:
        return np.full_like(quantiles, np.nan, dtype=float)

    sorter = np.argsort(values)
    values = values[sorter]
    sample_weight = sample_weight[sorter]
    cdf = np.cumsum(sample_weight)
    cdf = cdf / cdf[-1]
    return np.interp(quantiles, cdf, values)


def purity_stability(H):
    diag = np.diag(H)
    reco_sum = H.sum(axis=0)
    true_sum = H.sum(axis=1)
    with np.errstate(divide="ignore", invalid="ignore"):
        purity = np.where(reco_sum > 0, diag / reco_sum, 0.0)
        stability = np.where(true_sum > 0, diag / true_sum, 0.0)
    return purity, stability


def make_response(true_t, reco_t, weights, edges):
    H, _, _ = np.histogram2d(true_t, reco_t, bins=[edges, edges], weights=weights)
    return H


def all_rebinnings(base_edges, min_nbins, max_nbins):
    n_intermediate = len(base_edges) - 2
    min_keep = max(0, min_nbins - 1)
    max_keep = min(n_intermediate, max_nbins - 1)

    for nkeep in range(max_keep, min_keep - 1, -1):
        for combo in itertools.combinations(range(1, len(base_edges) - 1), nkeep):
            edges = [base_edges[0]] + [base_edges[i] for i in combo] + [base_edges[-1]]
            yield edges


def load_resolution_summary(path):
    if not path:
        return None
    path = os.path.expanduser(path)
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    needed = {"method", "t_low", "t_high", "sumw", "median_rel", "core_rel"}
    missing = needed - set(df.columns)
    if missing:
        raise ValueError(f"Resolution summary missing columns: {sorted(missing)}")
    df = df.copy()
    df["t_center"] = 0.5 * (df["t_low"] + df["t_high"])
    df["core_abs"] = df["core_rel"].abs() * df["t_center"]
    df["bias_abs"] = df["median_rel"].abs() * df["t_center"]
    return df


def aggregate_resolution_info(resolution_df, method, edges):
    if resolution_df is None:
        return {}

    mdf = resolution_df[resolution_df["method"] == method]
    out = {}
    bin_passes = []

    for i in range(len(edges) - 1):
        low, high = edges[i], edges[i + 1]
        # Include fine bins whose centers are inside the candidate bin.
        sub = mdf[(mdf["t_center"] >= low) & (mdf["t_center"] < high)].copy()
        width = high - low
        prefix = f"bin{i}_"

        if sub.empty:
            out[prefix + "raw_wavg_core_abs"] = np.nan
            out[prefix + "raw_wavg_bias_abs"] = np.nan
            out[prefix + "raw_max_core_abs"] = np.nan
            out[prefix + "raw_max_bias_abs"] = np.nan
            out[prefix + "resolution_width_ratio"] = np.nan
            out[prefix + "bias_width_ratio"] = np.nan
            bin_passes.append(False)
            continue

        w = np.asarray(sub["sumw"], dtype=float)
        w = np.where(np.isfinite(w) & (w > 0), w, 1.0)

        core_abs = np.asarray(sub["core_abs"], dtype=float)
        bias_abs = np.asarray(sub["bias_abs"], dtype=float)

        wavg_core = float(np.average(core_abs, weights=w))
        wavg_bias = float(np.average(bias_abs, weights=w))
        max_core = float(np.nanmax(core_abs))
        max_bias = float(np.nanmax(bias_abs))

        out[prefix + "raw_wavg_core_abs"] = wavg_core
        out[prefix + "raw_wavg_bias_abs"] = wavg_bias
        out[prefix + "raw_max_core_abs"] = max_core
        out[prefix + "raw_max_bias_abs"] = max_bias
        out[prefix + "resolution_width_ratio"] = width / wavg_core if wavg_core > 0 else np.inf
        out[prefix + "bias_width_ratio"] = width / wavg_bias if wavg_bias > 0 else np.inf
        bin_passes.append(True)

    # Compact summary over candidate bins
    core_ratios = [v for k, v in out.items() if k.endswith("resolution_width_ratio") and np.isfinite(v)]
    bias_ratios = [v for k, v in out.items() if k.endswith("bias_width_ratio") and np.isfinite(v)]
    out["min_resolution_width_ratio_raw"] = float(np.min(core_ratios)) if core_ratios else np.nan
    out["min_bias_width_ratio_raw"] = float(np.min(bias_ratios)) if bias_ratios else np.nan
    return out


def build_calibration(df, mode, n_knots=12, train_fraction=1.0, seed=12345):
    """Return calibrated reco_t and a dataframe describing the calibration."""
    reco = df["reco_t_raw"].to_numpy()
    true = df["true_t"].to_numpy()
    w = df["stat_weight"].to_numpy()

    if mode == "none":
        curve = pd.DataFrame({"mode": ["none"], "x_reco": [np.nan], "y_true": [np.nan]})
        return reco.copy(), curve

    finite = np.isfinite(reco) & np.isfinite(true) & np.isfinite(w) & (reco > 0) & (true > 0) & (w > 0)

    if train_fraction < 1.0:
        rng = np.random.default_rng(seed)
        train = rng.random(len(df)) < train_fraction
        finite = finite & train

    if np.sum(finite) < 10:
        curve = pd.DataFrame({"mode": [mode], "x_reco": [np.nan], "y_true": [np.nan]})
        return reco.copy(), curve

    r_train = reco[finite]
    t_train = true[finite]
    w_train = w[finite]

    if mode == "scale":
        ratio = t_train / r_train
        scale = float(weighted_quantile(ratio, [0.5], w_train)[0])
        reco_cal = reco * scale
        curve = pd.DataFrame({"mode": ["scale"], "x_reco": [1.0], "y_true": [scale]})
        return reco_cal, curve

    if mode == "quantile":
        probs = np.linspace(0.02, 0.98, n_knots)
        xk = weighted_quantile(r_train, probs, w_train)
        yk = weighted_quantile(t_train, probs, w_train)

        # Ensure strictly increasing x knots for np.interp.
        keep = np.isfinite(xk) & np.isfinite(yk)
        xk = xk[keep]
        yk = yk[keep]
        probs = probs[keep]

        uniq_x, uniq_idx = np.unique(xk, return_index=True)
        xk = uniq_x
        yk = yk[uniq_idx]
        probs = probs[uniq_idx]

        if len(xk) < 2:
            curve = pd.DataFrame({"mode": ["quantile"], "prob": [np.nan], "x_reco": [np.nan], "y_true": [np.nan]})
            return reco.copy(), curve

        reco_cal = np.interp(reco, xk, yk, left=yk[0], right=yk[-1])
        curve = pd.DataFrame({"mode": "quantile", "prob": probs, "x_reco": xk, "y_true": yk})
        return reco_cal, curve

    raise ValueError(f"Unknown calibration mode: {mode}")


def evaluate_edges(df, edges, min_ps, min_yield, method=None, calibration="none", resolution_df=None, resolution_width_factor=None, bias_width_factor=None):
    true_t = df["true_t"].to_numpy()
    reco_t = df["reco_t"].to_numpy()
    w_stat = df["stat_weight"].to_numpy()
    w_corr = df["corr_weight"].to_numpy()

    H_stat = make_response(true_t, reco_t, w_stat, edges)
    H_corr = make_response(true_t, reco_t, w_corr, edges)

    pur, stab = purity_stability(H_stat)
    diag = np.diag(H_stat)
    true_yield_stat = H_stat.sum(axis=1)
    reco_yield_stat = H_stat.sum(axis=0)
    true_yield_corr = H_corr.sum(axis=1)
    reco_yield_corr = H_corr.sum(axis=0)

    rows = []
    for i in range(len(edges) - 1):
        rows.append({
            "bin": i,
            "t_low": edges[i],
            "t_high": edges[i + 1],
            "purity": pur[i],
            "stability": stab[i],
            "diag_yield_stat": diag[i],
            "true_yield_stat": true_yield_stat[i],
            "reco_yield_stat": reco_yield_stat[i],
            "true_yield_corr": true_yield_corr[i],
            "reco_yield_corr": reco_yield_corr[i],
            "passes_bin_ps_yield": (
                pur[i] >= min_ps and
                stab[i] >= min_ps and
                true_yield_stat[i] >= min_yield and
                reco_yield_stat[i] >= min_yield
            ),
        })
    detail = pd.DataFrame(rows)

    in_true_range = np.sum(w_stat[(true_t >= edges[0]) & (true_t < edges[-1])])
    total_stat = np.sum(w_stat)
    coverage_stat = float(in_true_range / total_stat) if total_stat > 0 else 0.0

    if len(detail) == 0:
        min_ps_val = 0.0
        min_yield_val = 0.0
        passes_ps_yield = False
        failing = 0
    else:
        min_ps_val = float(np.min(np.minimum(detail["purity"], detail["stability"])))
        min_yield_val = float(np.min(np.minimum(detail["true_yield_stat"], detail["reco_yield_stat"])))
        passes_ps_yield = bool(detail["passes_bin_ps_yield"].all())
        failing = int((~detail["passes_bin_ps_yield"]).sum())

    summary = {
        "method": method,
        "calibration": calibration,
        "edges": ",".join(f"{x:g}" for x in edges),
        "nbins": len(edges) - 1,
        "passes_ps_yield": passes_ps_yield,
        "failing_bins_ps_yield": failing,
        "min_purity_stability": min_ps_val,
        "min_yield_stat": min_yield_val,
        "coverage_stat": coverage_stat,
        "sum_stat_in_range": float(H_stat.sum()),
        "sum_corr_in_range": float(H_corr.sum()),
    }

    res_info = aggregate_resolution_info(resolution_df, method, edges)
    summary.update(res_info)

    passes_resolution = True
    if resolution_width_factor is not None and np.isfinite(summary.get("min_resolution_width_ratio_raw", np.nan)):
        passes_resolution = passes_resolution and (summary["min_resolution_width_ratio_raw"] >= resolution_width_factor)
    if bias_width_factor is not None and np.isfinite(summary.get("min_bias_width_ratio_raw", np.nan)):
        passes_resolution = passes_resolution and (summary["min_bias_width_ratio_raw"] >= bias_width_factor)

    summary["passes_resolution_raw"] = bool(passes_resolution)
    summary["passes"] = bool(passes_ps_yield and passes_resolution)

    return summary, detail, H_stat


def score_summary(s, min_ps, min_yield):
    ps_ratio = s["min_purity_stability"] / min_ps if min_ps > 0 else 1.0
    y_ratio = s["min_yield_stat"] / min_yield if min_yield > 0 else 1.0
    quality = min(ps_ratio, y_ratio)

    # If raw resolution summary is available, include a soft preference for wider-than-resolution bins.
    res_ratio = s.get("min_resolution_width_ratio_raw", np.nan)
    if np.isfinite(res_ratio):
        quality = min(quality, max(0.0, res_ratio / 2.0))

    return (
        int(bool(s.get("passes", False))),
        int(bool(s.get("passes_ps_yield", False))),
        s["nbins"],
        quality,
        s["min_purity_stability"],
        s["coverage_stat"],
        s["min_yield_stat"],
    )


def load_method_dataframe(tree_path, tree_name, method, args):
    cfg = METHODS[method]
    cuts = [
        "true_t > 0",
        f"{cfg['reco']} > 0",
        f"{cfg['stat_weight']} > 0",
        "reco_mass >= %.12g" % args.mass_min,
        "reco_mass <= %.12g" % args.mass_max,
        "abs(sigma_epz - %.12g) <= %.12g" % (2.0 * args.electron_energy, args.sigma_epz_half_width),
        "reco_W >= %.12g" % args.w_min,
    ]
    if cfg["extra_cut"]:
        cuts.append(cfg["extra_cut"])

    cut = " && ".join(cuts)
    cols = sorted(set(["true_t", cfg["reco"], cfg["stat_weight"], cfg["corr_weight"]]))

    rdf = ROOT.RDataFrame(tree_name, tree_path).Filter(cut)
    arrs = rdf.AsNumpy(columns=cols)
    if len(arrs) == 0 or len(next(iter(arrs.values()))) == 0:
        return pd.DataFrame(columns=["true_t", "reco_t_raw", "stat_weight", "corr_weight"])

    df = pd.DataFrame({
        "true_t": np.asarray(arrs["true_t"], dtype=float),
        "reco_t_raw": np.asarray(arrs[cfg["reco"]], dtype=float),
        "stat_weight": np.asarray(arrs[cfg["stat_weight"]], dtype=float) * args.yield_scale,
        "corr_weight": np.asarray(arrs[cfg["corr_weight"]], dtype=float) * args.yield_scale,
    })

    df = df.replace([np.inf, -np.inf], np.nan).dropna()
    return df


def method_diagnostics(df, method, calibration):
    if len(df) == 0:
        return {
            "method": method,
            "calibration": calibration,
            "n_entries": 0,
            "sum_stat_weight": 0.0,
            "sum_corr_weight": 0.0,
            "corr_linear": np.nan,
            "corr_log1p": np.nan,
            "median_delta": np.nan,
            "median_ratio": np.nan,
        }

    t = df["true_t"].to_numpy()
    r = df["reco_t"].to_numpy()
    corr_linear = np.corrcoef(t, r)[0, 1] if len(df) > 1 else np.nan
    corr_log = np.corrcoef(np.log1p(t), np.log1p(r))[0, 1] if len(df) > 1 else np.nan
    delta = r - t
    ratio = np.where(t > 0, r / t, np.nan)

    return {
        "method": method,
        "calibration": calibration,
        "n_entries": int(len(df)),
        "sum_stat_weight": float(np.sum(df["stat_weight"])),
        "sum_corr_weight": float(np.sum(df["corr_weight"])),
        "corr_linear": float(corr_linear),
        "corr_log1p": float(corr_log),
        "median_delta": float(np.nanmedian(delta)),
        "q16_delta": float(np.nanquantile(delta, 0.16)),
        "q84_delta": float(np.nanquantile(delta, 0.84)),
        "median_ratio": float(np.nanmedian(ratio)),
        "q16_ratio": float(np.nanquantile(ratio, 0.16)),
        "q84_ratio": float(np.nanquantile(ratio, 0.84)),
    }


def main():
    p = argparse.ArgumentParser(description="Resolution-aware aggressive |t| binning across all Upsilon t methods.")
    
    # Changed tuple from a required positional arg to an optional arg for Jupyter
    p.add_argument("--tuple", default="~/merged_elecs_test_tuples_fix.root", help="Input ROOT file with UpsilonTree")
    p.add_argument("--tree-name", default="UpsilonTree")
    p.add_argument("--outdir", default="t_binning_resolution_aware_v4")
    p.add_argument("--electron-energy", type=float, default=18.0)
    p.add_argument("--sigma-epz-half-width", type=float, default=None)
    p.add_argument("--mass-min", type=float, default=8.0)
    p.add_argument("--mass-max", type=float, default=12.0)
    p.add_argument("--w-min", type=float, default=20.0)
    p.add_argument("--yield-scale", type=float, default=1.0)
    p.add_argument("--min-purity-stability", type=float, default=0.35)
    p.add_argument("--min-stat-yield", type=float, default=20.0)
    p.add_argument("--min-nbins", type=int, default=3)
    p.add_argument("--max-nbins", type=int, default=7)
    p.add_argument("--base-edges", default=",".join(str(x) for x in BASE_EDGES_DEFAULT))
    p.add_argument("--resolution-summary", default="", help="Optional t_resolution_summary.csv")
    p.add_argument("--calibrations", default="none,scale,quantile", help="Comma-separated list: none,scale,quantile")
    p.add_argument("--n-quantile-knots", type=int, default=12)
    p.add_argument("--calibration-train-fraction", type=float, default=1.0)
    p.add_argument("--seed", type=int, default=12345)
    p.add_argument("--resolution-width-factor", type=float, default=None, help="Optional hard constraint: bin width / raw core_abs >= this")
    p.add_argument("--bias-width-factor", type=float, default=None, help="Optional hard constraint: bin width / raw abs bias >= this")
    
    # JUPYTER FIX: Detect notebook environment and bypass hidden parameters
    if "ipykernel" in sys.modules:
        args = p.parse_args([])
    else:
        args = p.parse_args()

    if args.sigma_epz_half_width is None:
        args.sigma_epz_half_width = 0.2 * args.electron_energy

    tree_path = os.path.expanduser(args.tuple)
    os.makedirs(args.outdir, exist_ok=True)

    base_edges = parse_edges(args.base_edges)
    calibration_modes = [x.strip() for x in args.calibrations.split(",") if x.strip()]
    resolution_df = load_resolution_summary(args.resolution_summary)

    print("=== Resolution-aware |t| binning optimizer ===")
    print(f"Input tuple: {tree_path}")
    print(f"Selection: mass [{args.mass_min}, {args.mass_max}], |SigmaEPz - {2*args.electron_energy:g}| <= {args.sigma_epz_half_width:g}, W >= {args.w_min:g}")
    print(f"Requirements: purity/stability >= {args.min_purity_stability:g}, stat yield >= {args.min_stat_yield:g}, nbins in [{args.min_nbins}, {args.max_nbins}]")
    print(f"Calibration modes: {calibration_modes}")
    if resolution_df is not None:
        print(f"Using resolution summary: {os.path.expanduser(args.resolution_summary)}")
    print("")

    all_scan_rows = []
    best_rows = []
    diagnostics_rows = []
    fixed_summary_rows = []
    fixed_detail_rows = []

    for method in METHODS:
        cfg = METHODS[method]
        print(f"--- {method}: {cfg['label']} ---")

        raw_df = load_method_dataframe(tree_path, args.tree_name, method, args)
        if len(raw_df) == 0:
            print("  No selected events.")
            continue

        for calib in calibration_modes:
            reco_cal, calib_curve = build_calibration(
                raw_df, calib,
                n_knots=args.n_quantile_knots,
                train_fraction=args.calibration_train_fraction,
                seed=args.seed,
            )

            df = raw_df.copy()
            df["reco_t"] = reco_cal
            calib_curve.insert(0, "method", method)
            calib_curve.to_csv(os.path.join(args.outdir, f"calibration_curve_{method}_{calib}.csv"), index=False)

            diag = method_diagnostics(df, method, calib)
            diagnostics_rows.append(diag)

            print(f"  {calib:8s} entries={diag['n_entries']:6d}, sumw={diag['sum_stat_weight']:.1f}, corr(true,reco)={diag['corr_linear']:.3f}, median reco/true={diag['median_ratio']:.3f}")

            best_summary = None
            best_detail = None
            best_H = None

            for edges in all_rebinnings(base_edges, args.min_nbins, args.max_nbins):
                summary, detail, H = evaluate_edges(
                    df, edges,
                    args.min_purity_stability,
                    args.min_stat_yield,
                    method=method,
                    calibration=calib,
                    resolution_df=resolution_df,
                    resolution_width_factor=args.resolution_width_factor,
                    bias_width_factor=args.bias_width_factor,
                )
                summary["label"] = cfg["label"]
                summary["method_preference"] = cfg["preference"]
                all_scan_rows.append(summary)

                if best_summary is None or score_summary(summary, args.min_purity_stability, args.min_stat_yield) > score_summary(best_summary, args.min_purity_stability, args.min_stat_yield):
                    best_summary = summary
                    best_detail = detail.copy()
                    best_H = H

            if best_summary is not None:
                best_rows.append(best_summary)
                best_detail.insert(0, "calibration", calib)
                best_detail.insert(0, "method", method)
                best_detail.to_csv(os.path.join(args.outdir, f"best_bins_{method}_{calib}.csv"), index=False)
                np.savetxt(os.path.join(args.outdir, f"migration_best_{method}_{calib}.csv"), best_H, delimiter=",")

                status = "PASS" if best_summary["passes"] else "FAIL"
                print(f"    best {status}: nbins={best_summary['nbins']}, edges={best_summary['edges']}, minPS={best_summary['min_purity_stability']:.4f}")

            for cname, edges in FIXED_CANDIDATES.items():
                summary, detail, H = evaluate_edges(
                    df, edges,
                    args.min_purity_stability,
                    args.min_stat_yield,
                    method=method,
                    calibration=calib,
                    resolution_df=resolution_df,
                    resolution_width_factor=args.resolution_width_factor,
                    bias_width_factor=args.bias_width_factor,
                )
                summary["candidate"] = cname
                fixed_summary_rows.append(summary)
                dd = detail.copy()
                dd.insert(0, "candidate", cname)
                dd.insert(0, "calibration", calib)
                dd.insert(0, "method", method)
                fixed_detail_rows.append(dd)

        print("")

    scan_df = pd.DataFrame(all_scan_rows)
    best_df = pd.DataFrame(best_rows)
    diagnostics_df = pd.DataFrame(diagnostics_rows)
    fixed_summary_df = pd.DataFrame(fixed_summary_rows)
    fixed_detail_df = pd.concat(fixed_detail_rows, ignore_index=True) if fixed_detail_rows else pd.DataFrame()

    scan_df.to_csv(os.path.join(args.outdir, "scan_all_methods_all_calibrations.csv"), index=False)
    best_df.to_csv(os.path.join(args.outdir, "best_method_summary.csv"), index=False)
    diagnostics_df.to_csv(os.path.join(args.outdir, "method_diagnostics.csv"), index=False)
    fixed_summary_df.to_csv(os.path.join(args.outdir, "fixed_candidate_binnings_summary.csv"), index=False)
    fixed_detail_df.to_csv(os.path.join(args.outdir, "fixed_candidate_binnings_detail.csv"), index=False)

    if len(best_df) == 0:
        print("No method/calibration combinations available.")
        return

    passing = best_df[best_df["passes"] == True].copy()
    if len(passing) > 0:
        # Prefer more bins, higher minimum purity/stability, and standard hybrid methods.
        passing = passing.sort_values(
            by=["nbins", "min_purity_stability", "coverage_stat", "method_preference"],
            ascending=[False, False, False, False],
        )
        rec = passing.iloc[0].to_dict()
        reason = "passes requested purity/stability, yield, and optional resolution constraints"
    else:
        tmp = best_df.copy()
        tmp["_score"] = tmp.apply(lambda r: score_summary(r, args.min_purity_stability, args.min_stat_yield), axis=1)
        tmp = tmp.sort_values(
            by=["nbins", "min_purity_stability", "coverage_stat", "min_yield_stat", "method_preference"],
            ascending=[False, False, False, False],
        )
        rec = tmp.iloc[0].to_dict()
        reason = "NO method/calibration passes requested thresholds. This is the least-bad diagnostic binning, not a final physics recommendation."

    rec_df = pd.DataFrame([rec])
    rec_df["reason"] = reason
    rec_df.to_csv(os.path.join(args.outdir, "recommended_method_and_bins.csv"), index=False)

    # Always write a theory request grid that is safe to ask for even when the experimental binning is not final.
    theory_bins = pd.DataFrame({
        "bin_low": [0.00, 0.25, 0.50, 0.75, 1.00, 1.50],
        "bin_high": [0.25, 0.50, 0.75, 1.00, 1.50, 2.00],
        "comment": ["fine theory grid, rebin after detector response validation"] * 6,
    })
    theory_bins.to_csv(os.path.join(args.outdir, "theory_bins_to_request_fine_grid.csv"), index=False)

    print("=== Best method summary ===")
    cols = ["method", "calibration", "passes", "passes_ps_yield", "nbins", "edges", "min_purity_stability", "min_yield_stat", "coverage_stat", "sum_stat_in_range", "sum_corr_in_range"]
    if len(best_df) > 0:
        print(best_df[cols].sort_values(by=["passes", "nbins", "min_purity_stability"], ascending=[False, False, False]).to_string(index=False))
    print("")

    print("=== Recommendation status ===")
    print(f"Method: {rec['method']}")
    print(f"Calibration: {rec['calibration']}")
    print(f"Edges: {rec['edges']}")
    print(f"Passes requirements: {rec['passes']}")
    print(f"Reason: {reason}")
    print("")
    print(f"Files written to: {args.outdir}")
    print("Key outputs:")
    print("  method_diagnostics.csv")
    print("  best_method_summary.csv")
    print("  recommended_method_and_bins.csv")
    print("  theory_bins_to_request_fine_grid.csv")
    print("  fixed_candidate_binnings_summary.csv")
    print("  fixed_candidate_binnings_detail.csv")
    print("  scan_all_methods_all_calibrations.csv")
    print("  best_bins_<method>_<calibration>.csv")
    print("  migration_best_<method>_<calibration>.csv")
    print("  calibration_curve_<method>_<calibration>.csv")


if __name__ == "__main__":
    main()